# VoiceGuard AI — Phase 3c: teach the voice detector to catch XTTS clones

Extends Phase 3b: real speech (AI4Bharat) vs fakes (MMS-TTS **+ XTTS clones** + ASVspoof slice),
with augmentation. XTTS clones are added so the model learns the fingerprint of the exact
cloner used in the demo.

**Setup:** Kaggle notebook · **GPU T4** · **Internet On** · Add-ons → **Secrets** → `HF_TOKEN` ·
Add Input → `awsaf49/asvpoof-2019-dataset`.

**Run:** keep `DRY_RUN=True` first (~8 min sanity). For interactive dry-run, after Cell 1 do
**Runtime → Restart session**, then run Cell 2 onward. For the real run set `DRY_RUN=False` and
use **Save Version → Save & Run All (Commit)** (a fresh commit kernel needs no restart).

In [ ]:
# 1) Install the cloner + a transformers that works for BOTH XTTS and wav2vec2 training
!pip install -q coqui-tts "transformers>=4.57,<5"
import os
os.environ["COQUI_TOS_AGREED"] = "1"
print("If interactive: Runtime -> Restart session, then run from Cell 2. (Commit run: no restart.)")

In [ ]:
# 2) Imports + config + HF login
import os, itertools, random, tempfile, numpy as np, torch, librosa, soundfile as sf
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, VitsModel, AutoTokenizer
from sklearn.metrics import accuracy_score, roc_curve

DRY_RUN  = True
PER      = 20 if DRY_RUN else 600     # real clips per language
XTTS_PER = 15 if DRY_RUN else 300     # XTTS clones per language (slow ~2s each)
EPOCHS   = 1 if DRY_RUN else 3
ASV_N    = 40 if DRY_RUN else 800
SR = 16000; MAX_LEN = SR * 4
BASE_MODEL = "motheecreator/Deepfake-audio-detection"
ASV = "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA"

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; print("device:", DEVICE)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN")); print("HF OK")

In [ ]:
# 3) Load REAL speech (AI4Bharat Hindi + Indian English; FLEURS fallback)
def find_text(ex):
    for k in ("normalized", "text", "transcription", "sentence", "raw_transcription"):
        v = ex.get(k)
        if isinstance(v, str) and v.strip(): return v.strip()
    return ""

def load_real(lang):
    try:
        if lang == "hindi":
            return load_dataset("ai4bharat/IndicVoices", "hindi", split="valid", streaming=True)
        return load_dataset("ai4bharat/Svarah", split="test", streaming=True)
    except Exception as e:
        code = "hi_in" if lang == "hindi" else "en_us"
        print(f"  {lang}: AI4Bharat unavailable ({str(e)[:60]}); FLEURS {code}")
        return load_dataset("google/fleurs", code, split="train", streaming=True)

def to_wav(a):
    if isinstance(a, dict) and "array" in a:
        return np.asarray(a["array"], dtype="float32"), int(a["sampling_rate"])
    if hasattr(a, "get_all_samples"):
        s = a.get_all_samples(); d = s.data
        w = d.numpy() if hasattr(d, "numpy") else np.asarray(d)
        if w.ndim > 1: w = w.mean(axis=0)
        return w.astype("float32"), int(s.sample_rate)
    raise ValueError("unknown audio type")

def take_real(ds, n):
    out = []
    for ex in itertools.islice(ds, n * 4):
        a = ex.get("audio_filepath") or ex.get("audio")
        if a is None: continue
        try:
            w, sr = to_wav(a)
        except Exception:
            continue
        if sr != SR: w = librosa.resample(w, orig_sr=sr, target_sr=SR)
        if len(w) < SR: continue
        out.append((w[:MAX_LEN], find_text(ex)))
        if len(out) >= n: break
    return out

real_hi = take_real(load_real("hindi"), PER)
real_en = take_real(load_real("english"), PER)
print(f"real Hindi {len(real_hi)} | real English {len(real_en)}")

In [ ]:
# 4) Fakes #1 - MMS-TTS (fast, clean digital TTS)
def mms(texts, name, n):
    m = VitsModel.from_pretrained(name).to(DEVICE).eval(); tk = AutoTokenizer.from_pretrained(name)
    tsr = getattr(m.config, "sampling_rate", 16000); out = []
    for t in texts:
        if not t: continue
        try:
            with torch.no_grad():
                w = m(**tk(t, return_tensors="pt").to(DEVICE)).waveform[0].cpu().numpy().astype("float32")
            if tsr != SR: w = librosa.resample(w, orig_sr=tsr, target_sr=SR)
            out.append(w[:MAX_LEN])
        except Exception: continue
        if len(out) >= n: break
    return out

fake_hi = mms([t for _, t in real_hi], "facebook/mms-tts-hin", PER)
fake_en = mms([t for _, t in real_en], "facebook/mms-tts-eng", PER)
print(f"MMS fakes: hi {len(fake_hi)} | en {len(fake_en)}")

In [ ]:
# 5) Fakes #2 - XTTS CLONES (the whole point of Phase 3c). Clone each real speaker.
from TTS.api import TTS
xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)
XTTS_SR = 24000
ref_path = os.path.join(tempfile.gettempdir(), "xref.wav")

def xtts_clones(reals, lang, n):
    out = []
    for wav, text in reals:
        if not text or len(text) < 4: continue
        try:
            sf.write(ref_path, wav, SR)
            w = np.asarray(xtts.tts(text=text[:200], speaker_wav=ref_path, language=lang), dtype="float32")
            w = librosa.resample(w, orig_sr=XTTS_SR, target_sr=SR)
            out.append(w[:MAX_LEN])
        except Exception:
            continue
        if len(out) >= n: break
    return out

xtts_hi = xtts_clones(real_hi, "hi", XTTS_PER)
xtts_en = xtts_clones(real_en, "en", XTTS_PER)
print(f"XTTS clones: hi {len(xtts_hi)} | en {len(xtts_en)}")

In [ ]:
# 6) Augment + ASVspoof slice + assemble (real=1, fake=0)
def augment(w):
    w = w.copy() * np.random.uniform(0.6, 1.0)
    if random.random() < 0.7:
        snr = np.random.uniform(8, 30); p = np.mean(w**2) + 1e-9
        n = np.random.randn(len(w)).astype("float32"); n *= np.sqrt(p/(10**(snr/10)))/(np.std(n)+1e-9); w = w + n
    if random.random() < 0.6:
        lo = random.choice([8000, 11025]); w = librosa.resample(w, orig_sr=SR, target_sr=lo); w = librosa.resample(w, orig_sr=lo, target_sr=SR)
    return np.clip(w, -1, 1).astype("float32")

def asv_slice(n):
    prot = f"{ASV}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"; flac = f"{ASV}/ASVspoof2019_LA_train/flac"
    rows = []
    with open(prot) as f:
        for line in f:
            p = line.split()
            if len(p) >= 5: rows.append((os.path.join(flac, p[1]+".flac"), 1 if p[4]=="bonafide" else 0))
    real = [r for r in rows if r[1]==1]; fake = [r for r in rows if r[1]==0]
    random.shuffle(real); random.shuffle(fake); sel = real[:n//2] + fake[:n//2]; out = []
    for path, y in sel:
        w, _ = librosa.load(path, sr=SR, mono=True); out.append((w[:MAX_LEN], y))
    return out

data  = [(w,1) for w,_ in real_hi] + [(w,1) for w,_ in real_en]
data += [(w,0) for w in fake_hi] + [(w,0) for w in fake_en]
data += [(w,0) for w in xtts_hi] + [(w,0) for w in xtts_en]   # XTTS clones as fakes
data += asv_slice(ASV_N)
random.shuffle(data)
k = int(len(data)*0.85); train_d, val_d = data[:k], data[k:]
print(f"total {len(data)} | train {len(train_d)} | val {len(val_d)}")

In [ ]:
# 7) Train (base = motheecreator wav2vec2) + save
extractor = AutoFeatureExtractor.from_pretrained(BASE_MODEL)
class DS(Dataset):
    def __init__(s, rows, aug): s.rows = rows; s.aug = aug
    def __len__(s): return len(s.rows)
    def __getitem__(s, i):
        w, y = s.rows[i]
        if s.aug and random.random() < 0.6: w = augment(w)
        return w.astype("float32"), y
def collate(b):
    f = extractor([x[0] for x in b], sampling_rate=SR, return_tensors="pt", padding=True)
    return f, torch.tensor([x[1] for x in b])
tr = DataLoader(DS(train_d, True),  batch_size=8, shuffle=True,  collate_fn=collate, num_workers=2)
va = DataLoader(DS(val_d,  False), batch_size=8, shuffle=False, collate_fn=collate, num_workers=2)

model = AutoModelForAudioClassification.from_pretrained(BASE_MODEL).to(DEVICE)
if hasattr(model, "freeze_feature_encoder"): model.freeze_feature_encoder()
opt = torch.optim.AdamW(model.parameters(), lr=1e-5); crit = nn.CrossEntropyLoss(); scaler = torch.amp.GradScaler("cuda")
for ep in range(EPOCHS):
    model.train(); tot = 0
    for f, ys in tr:
        f = {k: v.to(DEVICE) for k, v in f.items()}; ys = ys.to(DEVICE); opt.zero_grad()
        with torch.amp.autocast("cuda"): loss = crit(model(**f).logits, ys)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
    print(f"epoch {ep+1}/{EPOCHS} loss {tot/max(1,len(tr)):.4f}")
OUT = "/kaggle/working/voiceguard-india-xtts"; model.save_pretrained(OUT); extractor.save_pretrained(OUT); print("saved ->", OUT)

In [ ]:
# 8) Evaluate (accuracy + EER) + a focused check on XTTS clones only
def scores(dl):
    model.eval(); P, Y = [], []
    with torch.no_grad():
        for f, ys in dl:
            f = {k: v.to(DEVICE) for k, v in f.items()}
            P += torch.softmax(model(**f).logits, -1)[:, 0].cpu().tolist(); Y += ys.tolist()
    return np.array(P), np.array(Y)
P, Y = scores(va); yf = (Y == 0).astype(int)
acc = accuracy_score(yf, (P >= 0.5).astype(int)); fpr, tpr, _ = roc_curve(yf, P); fnr = 1 - tpr
eer = (fpr[np.nanargmin(np.abs(fnr-fpr))] + fnr[np.nanargmin(np.abs(fnr-fpr))]) / 2
print(f"VAL acc {acc*100:.2f}%  EER {eer*100:.2f}%")

# how well does it now flag XTTS clones? (P(fake) should be high)
xc = xtts_hi + xtts_en
if xc:
    Px, _ = scores(DataLoader(DS([(w,0) for w in xc], False), batch_size=8, collate_fn=collate))
    print(f"XTTS clones flagged as AI: {(Px>=0.5).mean()*100:.1f}%  (avg P_fake {Px.mean():.2f})")